# PAPL-NCCF for Vietnamese Mispronunciation Detection & Diagnosis (VietMDD)

**Colab-ready functional reimplementation of Tu et al., Interspeech 2023**

This notebook implements **one paper variant only: PAPL-NCCF**. It is chosen because Table 3 of the paper reports the best MD&D F1 among the paper variants (28.06%) and the lowest FAR (20.85%).

Paper topology reproduced here:

- `81-dim acoustic features -> 2 CNN + 2 BiLSTM`
- `NCCF pitch -> 2 CNN + 2 BiLSTM`
- `Vietnamese wav2vec2 ASR embeddings -> 1 CNN + 1 BiLSTM`
- `Canonical phonemes -> embedding + BiLSTM -> K,V`
- `concat(acoustic, pitch, phonetic) -> dot-product attention over canonical sequence`
- `concat(context, query) -> linear -> CTC -> observed phoneme sequence`

The paper does not publish all channel sizes, hidden dimensions, kernels, dropout, batch size, or the exact VLSP2020 wav2vec2 checkpoint. Those are marked as **implementation choices**. The topology, inputs, 16 kHz audio, 20 ms frame shift, CTC objective, AdamW learning rate `1e-5`, and maximum `101` epochs follow the paper.

**Recommended Colab runtime:** GPU (T4/L4/A100). The frozen wav2vec2 phonetic embeddings are cached once so they are not recomputed every epoch.

In [ ]:
#@title 1. Install dependencies
!pip -q install -U "transformers>=4.45" "datasets>=3.0" "huggingface_hub>=0.25" "soundfile>=0.12" "librosa>=0.10.2" "scipy>=1.11" "pandas>=2.0" "tqdm>=4.66"


In [ ]:
#@title 2. Imports, configuration, reproducibility
import os, io, gc, math, random, json
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly, medfilt
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torchaudio
from datasets import load_dataset, Audio
from transformers import AutoProcessor, AutoModel

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))

# Paper settings
SAMPLE_RATE=16000
FRAME_SHIFT_MS=20.0
MAX_EPOCHS=101
LR=1e-5

# Implementation choices not specified in the paper
BATCH_SIZE=16
CACHE_BATCH_SIZE=8
NUM_WORKERS=2
DROPOUT=0.20
ACOUSTIC_CONV_DIM=128
ACOUSTIC_RNN_HIDDEN=128
PITCH_CONV_DIM=64
PITCH_RNN_HIDDEN=64
PHONETIC_CONV_DIM=128
PHONETIC_RNN_HIDDEN=128
LING_EMBED_DIM=128
LING_RNN_HIDDEN=256

CACHE_DIR=Path("/content/papl_vietmdd_cache"); CACHE_DIR.mkdir(parents=True,exist_ok=True)
W2V_MODEL_NAME="nguyenvulebinh/wav2vec2-base-vietnamese-250h"
FREEZE_W2V=True


## 3. Load VietMDD audio + official phoneme labels

The Hugging Face mirror is used only as a convenient audio host. **Canonical** and **Transcript** phoneme labels are loaded from the official VietMDD GitHub repository. The next cells verify that every audio file matches an official sample ID before training.

In [ ]:
#@title 3. Download official phoneme CSVs
!git clone -q --depth 1 https://github.com/VietMDDDataset/VietMDD.git /content/VietMDD_official || true
OFFICIAL_DIR=Path("/content/VietMDD_official")
split_csv={"train":OFFICIAL_DIR/"train.csv","validation":OFFICIAL_DIR/"dev.csv","test":OFFICIAL_DIR/"test_fix.csv"}
official={}
for split,path in split_csv.items():
    df=pd.read_csv(path)[["Path","Canonical","Transcript"]].copy()
    df["Path"]=df["Path"].astype(str)
    official[split]=df
    print(split,len(df),"rows")
assert len(official["train"])==3181 and len(official["validation"])==473 and len(official["test"])==612


In [ ]:
#@title 4. Load Hugging Face audio without decoding and attach official labels
hf0=load_dataset("doof-ferb/VietMDD")
hf={split:hf0[split].cast_column("audio",Audio(decode=False)) for split in ["train","validation","test"]}

def audio_sample_id(audio_blob):
    p=audio_blob.get("path")
    return Path(p).stem if p else None

def attach_official_labels(hf_split,label_df,split_name):
    label_map=label_df.set_index("Path")[["Canonical","Transcript"]].to_dict("index")
    rows=[]; missing=[]
    for i,ex in enumerate(hf_split):
        sid=audio_sample_id(ex["audio"])
        if sid is None or sid not in label_map:
            missing.append((i,sid)); continue
        lab=label_map[sid]
        rows.append({"hf_index":i,"sample_id":sid,"canonical":str(lab["Canonical"]).strip(),"transcript":str(lab["Transcript"]).strip()})
    print(f"{split_name}: matched {len(rows)}/{len(hf_split)}")
    if missing: print("First unmatched:",missing[:10])
    assert len(rows)==len(hf_split), "HF audio filenames did not fully match official VietMDD IDs; do not train on a partial alignment."
    return pd.DataFrame(rows)

meta={split:attach_official_labels(hf[split],official[split],split) for split in ["train","validation","test"]}
display(meta["train"].head())


In [ ]:
#@title 5. Inspect a real mispronunciation example
def split_phones(s): return [p for p in str(s).strip().split() if p]
mis=meta["train"][meta["train"]["canonical"]!=meta["train"]["transcript"]]
print("Mispronounced utterances in train:",len(mis),"/",len(meta["train"]))
ex=mis.iloc[0]
print("sample_id :",ex.sample_id)
print("canonical :",ex.canonical)
print("observed  :",ex.transcript)


In [ ]:
#@title 6. Build the VietMDD phone vocabulary
all_phones=set()
for split in ["train","validation","test"]:
    for col in ["canonical","transcript"]:
        for s in meta[split][col]: all_phones.update(split_phones(s))
phones=sorted(all_phones)
id2phone=["<blank>"]+phones
phone2id={p:i for i,p in enumerate(id2phone)}
BLANK_ID=0; VOCAB_SIZE=len(id2phone)
print("Vocabulary size incl. blank:",VOCAB_SIZE)
print(id2phone)
print("Tone tokens:",sorted([p for p in phones if p.startswith("_")]))
def encode_phones(s): return [phone2id[p] for p in split_phones(s)]
def decode_phone_ids(ids): return [id2phone[int(i)] for i in ids if int(i)!=BLANK_ID]


## 7. Frame-level features

Three synchronized 20-ms streams are used: 81-dim acoustic features, a one-dimensional NCCF-derived pitch contour, and frozen Vietnamese wav2vec2 phonetic embeddings.

In [ ]:
#@title 7. Audio decoding and feature extraction
def decode_audio_blob(blob,target_sr=SAMPLE_RATE):
    if blob.get("bytes") is not None:
        wav,sr=sf.read(io.BytesIO(blob["bytes"]),dtype="float32",always_2d=False)
    elif blob.get("path") is not None:
        wav,sr=sf.read(blob["path"],dtype="float32",always_2d=False)
    else: raise ValueError("Audio blob has neither bytes nor path")
    wav=np.asarray(wav,dtype=np.float32)
    if wav.ndim==2: wav=wav.mean(axis=1)
    if sr!=target_sr:
        g=math.gcd(int(sr),int(target_sr)); wav=resample_poly(wav,target_sr//g,sr//g).astype(np.float32); sr=target_sr
    peak=np.max(np.abs(wav)) if len(wav) else 0
    if peak>1: wav=wav/peak
    return wav.astype(np.float32),sr

def frame_signal(wav,frame_length,frame_shift):
    wav=np.asarray(wav,dtype=np.float32)
    if len(wav)<frame_length: wav=np.pad(wav,(0,frame_length-len(wav)))
    n=1+max(0,math.ceil((len(wav)-frame_length)/frame_shift))
    total=(n-1)*frame_shift+frame_length
    if len(wav)<total: wav=np.pad(wav,(0,total-len(wav)))
    shape=(n,frame_length); strides=(wav.strides[0]*frame_shift,wav.strides[0])
    return np.lib.stride_tricks.as_strided(wav,shape=shape,strides=strides).copy()

def acoustic_81(wav,sr=SAMPLE_RATE):
    x=torch.from_numpy(wav).float().unsqueeze(0)
    fb=torchaudio.compliance.kaldi.fbank(x,sample_frequency=sr,frame_length=25.0,frame_shift=FRAME_SHIFT_MS,num_mel_bins=80,use_energy=False,dither=0.0,snip_edges=False)
    flen=int(round(.025*sr)); fshift=int(round(FRAME_SHIFT_MS/1000*sr))
    frames=frame_signal(wav,flen,fshift)
    loge=torch.from_numpy(np.log(np.maximum((frames**2).sum(axis=1),1e-10)).astype(np.float32)).unsqueeze(-1)
    T=min(len(fb),len(loge)); feat=torch.cat([fb[:T],loge[:T]],dim=-1)
    mean=feat.mean(0,keepdim=True); std=feat.std(0,keepdim=True).clamp_min(1e-5)
    return (feat-mean)/std

def nccf_pitch(wav,sr=SAMPLE_RATE,min_f0=50.0,max_f0=500.0,voiced_threshold=.30):
    # NCCF + median smoothing. max_f0=500 is an implementation choice for child speech.
    frame_len=int(round(.025*sr)); frame_shift=int(round(FRAME_SHIFT_MS/1000*sr))
    frames=frame_signal(wav,frame_len,frame_shift); frames=frames-frames.mean(axis=1,keepdims=True)
    min_lag=max(1,int(sr/max_f0)); max_lag=min(frame_len-2,int(sr/min_f0))
    f0=np.zeros(len(frames),dtype=np.float32)
    for i,frame in enumerate(frames):
        best_c=-1.; best_lag=0
        for lag in range(min_lag,max_lag+1):
            a=frame[:-lag]; b=frame[lag:]
            den=np.sqrt(np.dot(a,a)*np.dot(b,b))+1e-8; cc=float(np.dot(a,b)/den)
            if cc>best_c: best_c=cc; best_lag=lag
        if best_c>=voiced_threshold and best_lag>0: f0[i]=sr/best_lag
    if len(f0)>=3: f0=medfilt(f0,kernel_size=3).astype(np.float32)
    voiced=f0>0; out=np.zeros_like(f0)
    if voiced.any():
        z=np.log(f0[voiced]); out[voiced]=(z-z.mean())/max(z.std(),1e-5)
    return torch.from_numpy(out).unsqueeze(-1)

def resize_time(x,target_T):
    if x.shape[0]==target_T: return x
    return F.interpolate(x.T.unsqueeze(0),size=target_T,mode="linear",align_corners=False).squeeze(0).T.contiguous()


In [ ]:
#@title 8. Load Vietnamese wav2vec2 ASR as frozen phonetic feature extractor
processor=AutoProcessor.from_pretrained(W2V_MODEL_NAME)
w2v=AutoModel.from_pretrained(W2V_MODEL_NAME).to(DEVICE).eval()
if FREEZE_W2V:
    for p in w2v.parameters(): p.requires_grad=False
PHONETIC_INPUT_DIM=int(w2v.config.hidden_size)
print(type(w2v).__name__,"hidden size =",PHONETIC_INPUT_DIM)


In [ ]:
#@title 9. Sanity check feature shapes
blob=hf["train"][int(meta["train"].iloc[0].hf_index)]["audio"]
wav,sr=decode_audio_blob(blob)
with torch.inference_mode():
    proc=processor(wav,sampling_rate=sr,return_tensors="pt")
    ph=w2v(proc.input_values.to(DEVICE)).last_hidden_state[0].cpu()
ac=resize_time(acoustic_81(wav,sr),len(ph)); pi=resize_time(nccf_pitch(wav,sr),len(ph))
print("seconds",len(wav)/sr,"acoustic",tuple(ac.shape),"pitch",tuple(pi.shape),"phonetic",tuple(ph.shape))
assert ac.shape[0]==pi.shape[0]==ph.shape[0] and ac.shape[1]==81 and pi.shape[1]==1


## 10. Cache features once

Phonetic embeddings are cached as float16. On Colab this is much faster overall than running wav2vec2 again for each of 101 PAPL epochs.

In [ ]:
#@title 10. Precompute and cache all splits
def wav2vec_batch(waves):
    batch=processor(waves,sampling_rate=SAMPLE_RATE,return_tensors="pt",padding=True)
    iv=batch.input_values.to(DEVICE); am=getattr(batch,"attention_mask",None)
    if am is not None: am=am.to(DEVICE)
    with torch.inference_mode(): out=w2v(iv,attention_mask=am).last_hidden_state
    raw=torch.tensor([len(w) for w in waves],device=DEVICE)
    if hasattr(w2v,"_get_feat_extract_output_lengths"):
        lens=w2v._get_feat_extract_output_lengths(raw).long()
    else: lens=torch.clamp((raw/320.).ceil().long(),min=1,max=out.shape[1])
    return out.cpu(),lens.cpu()

def precompute_split(split):
    out_dir=CACHE_DIR/split; out_dir.mkdir(parents=True,exist_ok=True)
    m=meta[split].reset_index(drop=True).copy(); rec=[]
    for start in tqdm(range(0,len(m),CACHE_BATCH_SIZE),desc=f"Caching {split}"):
        chunk=m.iloc[start:start+CACHE_BATCH_SIZE]; waves=[]
        for row in chunk.itertuples():
            wav,sr=decode_audio_blob(hf[split][int(row.hf_index)]["audio"]); assert sr==SAMPLE_RATE; waves.append(wav)
        ph_batch,ph_lens=wav2vec_batch(waves)
        for j,row in enumerate(chunk.itertuples()):
            idx=start+j; path=out_dir/f"{idx:05d}.pt"
            if not path.exists():
                T=int(ph_lens[j]); ph=ph_batch[j,:T].contiguous(); ac=resize_time(acoustic_81(waves[j]),T); pi=resize_time(nccf_pitch(waves[j]),T)
                torch.save({"sample_id":row.sample_id,"acoustic":ac.half(),"pitch":pi.half(),"phonetic":ph.half(),"canonical_ids":torch.tensor(encode_phones(row.canonical),dtype=torch.long),"target_ids":torch.tensor(encode_phones(row.transcript),dtype=torch.long),"canonical_text":row.canonical,"target_text":row.transcript},path)
            rec.append({"cache_path":str(path),"sample_id":row.sample_id,"canonical":row.canonical,"transcript":row.transcript})
    man=pd.DataFrame(rec); man.to_csv(CACHE_DIR/f"{split}_manifest.csv",index=False); return man

manifests={split:precompute_split(split) for split in ["train","validation","test"]}
print("Cache complete:",CACHE_DIR)


In [ ]:
#@title 11. Release frozen ASR model from GPU after caching
del w2v; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("wav2vec2 released")


In [ ]:
#@title 12. Cached Dataset/DataLoader
from torch.utils.data import Dataset,DataLoader
class CachedPAPLDataset(Dataset):
    def __init__(self,manifest): self.manifest=manifest.reset_index(drop=True)
    def __len__(self): return len(self.manifest)
    def __getitem__(self,i): return torch.load(self.manifest.iloc[i].cache_path,map_location="cpu",weights_only=False)

def pad_time(xs):
    lens=torch.tensor([x.shape[0] for x in xs],dtype=torch.long); T=int(lens.max()); d=xs[0].shape[1]
    out=torch.zeros(len(xs),T,d,dtype=torch.float32)
    for i,x in enumerate(xs): out[i,:x.shape[0]]=x.float()
    return out,lens

def pad_phones(xs):
    lens=torch.tensor([len(x) for x in xs],dtype=torch.long); N=int(lens.max()); out=torch.full((len(xs),N),BLANK_ID,dtype=torch.long)
    for i,x in enumerate(xs): out[i,:len(x)]=x
    return out,lens

def papl_collate(batch):
    ac,fl=pad_time([x["acoustic"] for x in batch]); pi,pl=pad_time([x["pitch"] for x in batch]); ph,hl=pad_time([x["phonetic"] for x in batch])
    assert torch.equal(fl,pl) and torch.equal(fl,hl)
    can,cl=pad_phones([x["canonical_ids"] for x in batch]); tl=torch.tensor([len(x["target_ids"]) for x in batch],dtype=torch.long); targets=torch.cat([x["target_ids"] for x in batch])
    return {"acoustic":ac,"pitch":pi,"phonetic":ph,"frame_lengths":fl,"canonical":can,"canonical_lengths":cl,"targets":targets,"target_lengths":tl,"sample_ids":[x["sample_id"] for x in batch],"canonical_text":[x["canonical_text"] for x in batch],"target_text":[x["target_text"] for x in batch]}

train_ds=CachedPAPLDataset(manifests["train"]); dev_ds=CachedPAPLDataset(manifests["validation"]); test_ds=CachedPAPLDataset(manifests["test"])
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=papl_collate)
dev_loader=DataLoader(dev_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=papl_collate)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=papl_collate)
b=next(iter(train_loader)); print({k:tuple(b[k].shape) for k in ["acoustic","pitch","phonetic","canonical"]})


## 13. PAPL model

The paper specifies the number of CNN/RNN stacks but not their dimensions. This implementation uses time-preserving 1-D CNN stacks and bidirectional LSTMs, then implements the paper's dot-product linguistic attention equations directly.

In [ ]:
#@title 13. Model definition
class ConvStack(nn.Module):
    def __init__(self,in_dim,hid,n,drop):
        super().__init__(); layers=[]; d=in_dim
        for _ in range(n): layers += [nn.Conv1d(d,hid,3,padding=1),nn.BatchNorm1d(hid),nn.ReLU(),nn.Dropout(drop)]; d=hid
        self.net=nn.Sequential(*layers); self.out_dim=hid
    def forward(self,x): return self.net(x.transpose(1,2)).transpose(1,2)

class BiLSTMStack(nn.Module):
    def __init__(self,in_dim,hid,n,drop):
        super().__init__(); self.layers=nn.ModuleList(); self.norms=nn.ModuleList(); self.drops=nn.ModuleList(); d=in_dim
        for _ in range(n):
            self.layers.append(nn.LSTM(d,hid,batch_first=True,bidirectional=True)); self.norms.append(nn.LayerNorm(hid*2)); self.drops.append(nn.Dropout(drop)); d=hid*2
        self.out_dim=d
    def forward(self,x,lens):
        for rnn,norm,drop in zip(self.layers,self.norms,self.drops):
            p=pack_padded_sequence(x,lens.cpu(),batch_first=True,enforce_sorted=False); p,_=rnn(p); x,_=pad_packed_sequence(p,batch_first=True); x=drop(norm(x))
        return x

class FeatureEncoder(nn.Module):
    def __init__(self,in_dim,conv_dim,n_conv,rnn_hid,n_rnn,drop):
        super().__init__(); self.conv=ConvStack(in_dim,conv_dim,n_conv,drop); self.rnn=BiLSTMStack(self.conv.out_dim,rnn_hid,n_rnn,drop); self.out_dim=self.rnn.out_dim
    def forward(self,x,lens): return self.rnn(self.conv(x),lens)

class LinguisticEncoder(nn.Module):
    def __init__(self,vocab,emb,hid,qdim,drop):
        super().__init__(); self.emb=nn.Embedding(vocab,emb,padding_idx=BLANK_ID); self.rnn=nn.LSTM(emb,hid,batch_first=True,bidirectional=True); self.drop=nn.Dropout(drop); self.to_k=nn.Linear(hid*2,qdim); self.to_v=nn.Linear(hid*2,qdim)
    def forward(self,ids,lens):
        x=self.emb(ids); p=pack_padded_sequence(x,lens.cpu(),batch_first=True,enforce_sorted=False); p,_=self.rnn(p); x,_=pad_packed_sequence(p,batch_first=True); x=self.drop(x); return self.to_k(x),self.to_v(x)

class PAPL_NCCF(nn.Module):
    def __init__(self,vocab,ph_in):
        super().__init__()
        self.acoustic_encoder=FeatureEncoder(81,ACOUSTIC_CONV_DIM,2,ACOUSTIC_RNN_HIDDEN,2,DROPOUT)
        self.pitch_encoder=FeatureEncoder(1,PITCH_CONV_DIM,2,PITCH_RNN_HIDDEN,2,DROPOUT)
        self.phonetic_encoder=FeatureEncoder(ph_in,PHONETIC_CONV_DIM,1,PHONETIC_RNN_HIDDEN,1,DROPOUT)
        self.query_dim=self.acoustic_encoder.out_dim+self.phonetic_encoder.out_dim+self.pitch_encoder.out_dim
        self.linguistic_encoder=LinguisticEncoder(vocab,LING_EMBED_DIM,LING_RNN_HIDDEN,self.query_dim,DROPOUT)
        self.output=nn.Sequential(nn.Dropout(DROPOUT),nn.Linear(self.query_dim*2,vocab))
    def forward(self,acoustic,pitch,phonetic,frame_lengths,canonical,canonical_lengths,return_attention=False):
        ha=self.acoustic_encoder(acoustic,frame_lengths); hp=self.phonetic_encoder(phonetic,frame_lengths); hpi=self.pitch_encoder(pitch,frame_lengths)
        T=min(ha.shape[1],hp.shape[1],hpi.shape[1]); hq=torch.cat([ha[:,:T],hp[:,:T],hpi[:,:T]],dim=-1)
        hk,hv=self.linguistic_encoder(canonical,canonical_lengths)
        scores=torch.bmm(hq,hk.transpose(1,2)); N=hk.shape[1]
        mask=torch.arange(N,device=canonical_lengths.device)[None,:]>=canonical_lengths[:,None]
        scores=scores.masked_fill(mask[:,None,:],-1e4); alpha=torch.softmax(scores,dim=-1); context=torch.bmm(alpha,hv)
        logits=self.output(torch.cat([context,hq],dim=-1))
        return (logits,alpha) if return_attention else logits

model=PAPL_NCCF(VOCAB_SIZE,PHONETIC_INPUT_DIM).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


In [ ]:
#@title 14. CTC decoding + Needleman-Wunsch alignment
def ctc_greedy_decode(logits,lengths):
    pred=logits.argmax(-1).detach().cpu(); outs=[]
    for seq,L in zip(pred,lengths):
        seq=seq[:int(L)].tolist(); collapsed=[]; prev=None
        for x in seq:
            if x!=prev and x!=BLANK_ID: collapsed.append(x)
            prev=x
        outs.append(collapsed)
    return outs

def needleman_wunsch(ref,hyp):
    n,m=len(ref),len(hyp); dp=np.zeros((n+1,m+1),dtype=np.int32); bt=np.zeros((n+1,m+1),dtype=np.int8)
    for i in range(1,n+1): dp[i,0]=i; bt[i,0]=1
    for j in range(1,m+1): dp[0,j]=j; bt[0,j]=2
    for i in range(1,n+1):
        for j in range(1,m+1):
            vals=[dp[i-1,j-1]+(ref[i-1]!=hyp[j-1]),dp[i-1,j]+1,dp[i,j-1]+1]; k=int(np.argmin(vals)); dp[i,j]=vals[k]; bt[i,j]=k
    pairs=[]; i=n; j=m
    while i>0 or j>0:
        if i>0 and j>0 and bt[i,j]==0: pairs.append((ref[i-1],hyp[j-1])); i-=1; j-=1
        elif i>0 and (j==0 or bt[i,j]==1): pairs.append((ref[i-1],None)); i-=1
        else: pairs.append((None,hyp[j-1])); j-=1
    return pairs[::-1]

def edit_counts(ref,hyp):
    S=D=I=0
    for r,h in needleman_wunsch(ref,hyp):
        if r is None:I+=1
        elif h is None:D+=1
        elif r!=h:S+=1
    return S,D,I

def diagnose_alignment(canonical,predicted):
    pairs=needleman_wunsch(canonical,predicted); errors=[]; pos=-1
    for r,h in pairs:
        if r is not None: pos+=1
        if r==h: continue
        typ="insertion" if r is None else ("deletion" if h is None else "substitution")
        comp="tone" if ((r or "").startswith("_") or (h or "").startswith("_")) else "phoneme"
        errors.append({"canonical_position":pos if r is not None else pos+.5,"expected":r,"observed":h,"type":typ,"component":comp})
    return pairs,errors


In [ ]:
#@title 15. Training / validation utilities
ctc_loss_fn=nn.CTCLoss(blank=BLANK_ID,zero_infinity=True)
def move_batch(b):
    out=dict(b)
    for k in ["acoustic","pitch","phonetic","frame_lengths","canonical","canonical_lengths","targets","target_lengths"]: out[k]=out[k].to(DEVICE,non_blocking=True)
    return out

def forward_loss(model,b):
    logits=model(b["acoustic"],b["pitch"],b["phonetic"],b["frame_lengths"],b["canonical"],b["canonical_lengths"])
    inp=torch.clamp(b["frame_lengths"],max=logits.shape[1]); loss=ctc_loss_fn(logits.log_softmax(-1).transpose(0,1),b["targets"],inp,b["target_lengths"]); return loss,logits,inp

@torch.inference_mode()
def evaluate_phone_metrics(model,loader):
    model.eval(); loss_sum=0.; nb=0; S=D=I=N=0
    for cpu in loader:
        b=move_batch(cpu); loss,logits,lens=forward_loss(model,b); loss_sum+=float(loss); nb+=1
        hyps=ctc_greedy_decode(logits,lens); refs=[split_phones(s) for s in cpu["target_text"]]
        for ref,hids in zip(refs,hyps):
            s,d,i=edit_counts(ref,decode_phone_ids(hids)); S+=s; D+=d; I+=i; N+=len(ref)
    return {"loss":loss_sum/max(nb,1),"correctness":(N-S-D)/max(N,1),"accuracy":(N-S-D-I)/max(N,1),"PER":(S+D+I)/max(N,1),"S":S,"D":D,"I":I,"N":N}

optimizer=torch.optim.AdamW(model.parameters(),lr=LR)
scaler=torch.amp.GradScaler("cuda",enabled=torch.cuda.is_available())


In [ ]:
#@title 16. Train PAPL-NCCF (paper maximum = 101 epochs)
BEST_PATH="/content/papl_nccf_vietmdd_best.pt"; history=[]; best=float("inf")
for epoch in range(1,MAX_EPOCHS+1):
    model.train(); running=0.
    pbar=tqdm(train_loader,desc=f"Epoch {epoch:03d}/{MAX_EPOCHS}")
    for step,cpu in enumerate(pbar,1):
        b=move_batch(cpu); optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu",dtype=torch.float16,enabled=torch.cuda.is_available()): loss,_,_=forward_loss(model,b)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); scaler.step(optimizer); scaler.update()
        running+=float(loss.detach()); pbar.set_postfix(train_loss=f"{running/step:.4f}")
    dev=evaluate_phone_metrics(model,dev_loader); row={"epoch":epoch,"train_loss":running/max(len(train_loader),1),**{f"dev_{k}":v for k,v in dev.items() if k in ["loss","correctness","accuracy","PER"]}}; history.append(row)
    print(f"epoch={epoch:03d} train={row['train_loss']:.4f} dev_loss={dev['loss']:.4f} dev_PER={dev['PER']:.4f} dev_acc={dev['accuracy']:.4f}")
    if dev["PER"]<best:
        best=dev["PER"]
        torch.save({"model_state":model.state_dict(),"phone2id":phone2id,"id2phone":id2phone,"config":{"variant":"PAPL-NCCF","sample_rate":SAMPLE_RATE,"frame_shift_ms":FRAME_SHIFT_MS,"wav2vec2":W2V_MODEL_NAME,"phonetic_input_dim":PHONETIC_INPUT_DIM},"epoch":epoch,"dev_metrics":dev},BEST_PATH)
        print(" -> saved best",BEST_PATH)
pd.DataFrame(history).to_csv("/content/papl_training_history.csv",index=False)
print("Best dev PER:",best)


In [ ]:
#@title 17. Load best checkpoint and evaluate official test split
ckpt=torch.load(BEST_PATH,map_location=DEVICE,weights_only=False); model.load_state_dict(ckpt["model_state"]); model.eval(); test_metrics=evaluate_phone_metrics(model,test_loader); print(json.dumps(test_metrics,indent=2))


## 18. Per-phone diagnosis

For the web app, the useful output is the alignment between the known canonical sequence and the model-recognized observed sequence. This exposes substitutions, deletions, insertions, and tone-token errors.

In [ ]:
#@title 18. Diagnose one test utterance
@torch.inference_mode()
def predict_cached_item(model,item):
    cpu=papl_collate([item]); b=move_batch(cpu); logits=model(b["acoustic"],b["pitch"],b["phonetic"],b["frame_lengths"],b["canonical"],b["canonical_lengths"]); return decode_phone_ids(ctc_greedy_decode(logits,b["frame_lengths"])[0])
chosen=next((test_ds[i] for i in range(len(test_ds)) if test_ds[i]["canonical_text"]!=test_ds[i]["target_text"]),test_ds[0])
pred=predict_cached_item(model,chosen); canonical=split_phones(chosen["canonical_text"]); actual=split_phones(chosen["target_text"]); pairs,det=diagnose_alignment(canonical,pred); _,gt=diagnose_alignment(canonical,actual)
print("Sample:",chosen["sample_id"]); print("\nCanonical:\n"," ".join(canonical)); print("\nHuman observed:\n"," ".join(actual)); print("\nModel predicted:\n"," ".join(pred)); print("\nDetected errors:"); display(pd.DataFrame(det)); print("\nGround-truth errors:"); display(pd.DataFrame(gt))


In [ ]:
#@title 19. Visualize canonical ↔ predicted phone alignment
rows=[]
for r,h in pairs:
    status="✓" if r==h else ("INSERT" if r is None else ("DELETE" if h is None else ("TONE" if r.startswith("_") or (h and h.startswith("_")) else "SUB")))
    rows.append({"expected":r or "∅","predicted":h or "∅","status":status})
display(pd.DataFrame(rows))


In [ ]:
#@title 20. Exact edit-event MDD metrics for the application
def event_set(can,obs):
    _,errs=diagnose_alignment(can,obs); return {(e["canonical_position"],e["expected"],e["observed"],e["type"]) for e in errs}
@torch.inference_mode()
def evaluate_mdd_events(model,dataset):
    TP=FP=FN=tTP=tFP=tFN=0
    for i in tqdm(range(len(dataset)),desc="MDD event eval"):
        item=dataset[i]; can=split_phones(item["canonical_text"]); actual=split_phones(item["target_text"]); pred=predict_cached_item(model,item); gt=event_set(can,actual); pr=event_set(can,pred)
        TP+=len(gt&pr); FP+=len(pr-gt); FN+=len(gt-pr)
        is_tone=lambda e:(e[1] or "").startswith("_") or (e[2] or "").startswith("_")
        gtt={e for e in gt if is_tone(e)}; prt={e for e in pr if is_tone(e)}; tTP+=len(gtt&prt); tFP+=len(prt-gtt); tFN+=len(gtt-prt)
    p=TP/max(TP+FP,1); r=TP/max(TP+FN,1); f=2*p*r/max(p+r,1e-12); tp=tTP/max(tTP+tFP,1); tr=tTP/max(tTP+tFN,1); tf=2*tp*tr/max(tp+tr,1e-12)
    return {"event_precision":p,"event_recall":r,"event_f1":f,"tone_event_precision":tp,"tone_event_recall":tr,"tone_event_f1":tf}
print(json.dumps(evaluate_mdd_events(model,test_ds),indent=2))


### Metric note

The previous cell is a transparent exact-edit-event metric useful for the application. Tu et al. report the APL hierarchical TA/TR/FA/FR and CD/DE protocol; exact numerical reproduction of those paper metrics also depends on matching the legacy evaluator's alignment conventions. This does **not** change the model training objective. For the web product, `diagnose_alignment()` already gives the individual phone errors needed by the UI.

In [ ]:
#@title 21. Export artifacts for a backend API
EXPORT_DIR=Path("/content/papl_export"); EXPORT_DIR.mkdir(exist_ok=True)
torch.save({"model_state":model.state_dict(),"phone2id":phone2id,"id2phone":id2phone,"config":ckpt["config"]},EXPORT_DIR/"papl_nccf_vietmdd.pt")
with open(EXPORT_DIR/"vocab.json","w",encoding="utf-8") as f: json.dump({"phone2id":phone2id,"id2phone":id2phone},f,ensure_ascii=False,indent=2)
print("Exported:",list(EXPORT_DIR.iterdir()))


## References and reproduction boundary

- Tuong Tu Huu, Viet Thanh Pham, Thi Thu Trang Nguyen, Thai Lai Dao. *Mispronunciation detection and diagnosis model for tonal language, applied to Vietnamese.* Interspeech 2023, DOI 10.21437/Interspeech.2023-364.
- Wenxuan Ye et al. *An Approach to Mispronunciation Detection and Diagnosis with Acoustic, Phonetic and Linguistic (APL) Embeddings.* ICASSP 2022.
- Official VietMDD: https://github.com/VietMDDDataset/VietMDD
- Audio mirror: https://huggingface.co/datasets/doof-ferb/VietMDD
- Vietnamese wav2vec2 checkpoint: https://huggingface.co/nguyenvulebinh/wav2vec2-base-vietnamese-250h

**Exact from Tu et al.:** 16 kHz; 20 ms shift; 80 fbank + 1 energy; 2 CNN + 2 RNN acoustic; same stack count for pitch; 1 CNN + 1 RNN phonetic encoder; wav2vec2 fine-tuned on VLSP2020 for phonetic embeddings; canonical embedding + BiLSTM + K/V; attention; CTC; AdamW `1e-5`; maximum 101 epochs; Needleman-Wunsch evaluation.

**Reconstructed choices because they are not published:** CNN widths/kernels, BiLSTM hidden sizes, dropout, batch size, exact downloadable VLSP wav2vec2 checkpoint replacement, normalization details, and child-speech NCCF maximum F0.